In [21]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd


In [ ]:
base_url= "https://books.toscrape.com/catalogue/page-1.html"


In [ ]:
raw_books = []
current_page = 1
url = base_url

while url and current_page < 4:
    resopnse = requests.get(url)

    soup = BeautifulSoup(resopnse.text, "html.parser")

    articles = soup.find_all("article", class_="product_pod")
    for article in articles:
        title = article.h3.a["title"]
        price = article.find("p", class_="price_color").text[2:]
        rating = article.find("p", class_="star-rating")["class"][1]
        imag_src = article.find("img")["src"]
        image_url = "https://books.toscrape.com/" + imag_src
        # image= urljoin(base_url, imag_src)

        img_content = requests.get(image_url).content
        with open("images/" + title[:50].replace(":", "") + ".jpg", "wb") as f:
            f.write(img_content)
        
        raw_books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "file_path": f"images/{title[:50].replace(':', '')}.jpg"
        })

        print("name" , title)
        print("price" , price)
        print("rating" , rating)
        print("image_url" , image_url)
        # print("image" , image)
        print("file_path" , f"images/{title[:50].replace(':', '')}.jpg")
        print("-------------")

    next_page = soup.find("li", class_="next")
    current_page += 1
    url = f"https://books.toscrape.com/catalogue/page-{current_page}.html" if next_page else None

In [ ]:
df = pd.DataFrame(raw_books)
df.to_csv("scraped_books.csv", index=False)